# 02 — Gauge Preprocessing

Cleans the BMD monthly rainfall CSV, standardizes column names and saves a
model-ready gauge table.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import pandas as pd
import numpy as np

gauge_files = sorted((RAW_DIR / "gauge").glob("*.csv"))
if not gauge_files:
    raise FileNotFoundError("No CSV file found in data/raw/gauge.")

gauge_path = gauge_files[0]
gauge = pd.read_csv(gauge_path)
print(f"Loaded: {gauge_path.name}")
display(gauge.head())
print(gauge.columns.tolist())

Loaded: bmd_monthly_rainfall_2017_2022.csv


,station_id,station_name,latitude,longitude,year,month,date,rainfall_mm
0,CL503,Chalna,22.6012,89.5195,2017,1,2017-01-01,0.0
1,CL503,Chalna,22.6012,89.5195,2017,2,2017-02-01,0.0
2,CL503,Chalna,22.6012,89.5195,2017,3,2017-03-01,345.0
3,CL503,Chalna,22.6012,89.5195,2017,4,2017-04-01,270.0
4,CL503,Chalna,22.6012,89.5195,2017,5,2017-05-01,783.0


['station_id', 'station_name', 'latitude', 'longitude', 'year', 'month', 'date', 'rainfall_mm']


In [3]:
def normalize_column_name(name: str) -> str:
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
    )

gauge.columns = [normalize_column_name(c) for c in gauge.columns]

aliases = {
    "station_name": "station",
    "station": "station",
    "lat": "latitude",
    "latitude": "latitude",
    "lon": "longitude",
    "long": "longitude",
    "longitude": "longitude",
    "rainfall": "rainfall_mm",
    "rainfall_mm": "rainfall_mm",
    "precipitation": "rainfall_mm",
    "year": "year",
    "month": "month",
}
gauge = gauge.rename(columns={c: aliases.get(c, c) for c in gauge.columns})

required = {"year", "month", "rainfall_mm"}
missing = required - set(gauge.columns)
if missing:
    raise ValueError(f"Required columns are missing: {sorted(missing)}")

for column in ["year", "month", "rainfall_mm", "latitude", "longitude"]:
    if column in gauge.columns:
        gauge[column] = pd.to_numeric(gauge[column], errors="coerce")

gauge = gauge.dropna(subset=["year", "month", "rainfall_mm"]).copy()
gauge["year"] = gauge["year"].astype(int)
gauge["month"] = gauge["month"].astype(int)
gauge = gauge[gauge["month"].between(1, 12)]
gauge = gauge[gauge["rainfall_mm"] >= 0]
gauge["date"] = pd.to_datetime(
    dict(year=gauge["year"], month=gauge["month"], day=1)
)
gauge = gauge.drop_duplicates().sort_values("date").reset_index(drop=True)

display(gauge.head())
print(gauge.isna().sum())

,station_id,station,latitude,longitude,year,month,date,rainfall_mm
0,CL503,Chalna,22.6012,89.5195,2017,1,2017-01-01,0.0
1,CL510,Khulna,22.8319,89.5500,2017,1,2017-01-01,0.0
2,CL504,Dumuria,22.8093,89.4145,2017,1,2017-01-01,0.0
3,CL509,Kapilmuni,22.6887,89.3088,2017,1,2017-01-01,0.0
4,CL515,Paikgacha,22.5850,89.3182,2017,1,2017-01-01,0.0


station_id     0
station        0
latitude       0
longitude      0
year           0
month          0
date           0
rainfall_mm    0
dtype: int64


In [4]:
output_dir = PROCESSED_DIR / "station_samples"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "gauge_monthly_clean.csv"
gauge.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_samples\gauge_monthly_clean.csv
